# 02 - QLoRA Fine-Tuning (Colab, T4/A100)
Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with 4-bit QLoRA using TRL's `SFTTrainer` on the 15k-example subsample from notebook 01.

**Runtime:** GPU (T4 is sufficient at 4-bit; A100 will be faster).

Upload `data/train.jsonl` and `data/val.jsonl` from notebook 01 before running, or re-run the data-prep cells here.

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets torchao

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUTPUT_DIR = 'text2sql-qwen2.5-1.5b-qlora'

In [ ]:
# If notebook 01 wasn't run in this session, upload data/train.jsonl + data/val.jsonl first.
dataset = load_dataset('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/val.jsonl'})

def format_example(example):
    return {'text': example['prompt'] + example['completion']}

dataset = dataset.map(format_example)
dataset['train'][0]['text'][:500]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,  # T4 (Turing) lacks native bf16 Tensor Core support; fp16 is much faster
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,  # keep non-quantized params (incl. LoRA adapters) in fp16, matching bnb_4bit_compute_dtype
    device_map='auto',
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)

# trl>=1.13 dropped SFTConfig(warmup_ratio=...) and renamed max_seq_length -> max_length;
# compute the equivalent warmup_steps (3% of total optimizer steps) explicitly.
NUM_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
WARMUP_RATIO = 0.03

effective_batch_size = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = -(-len(dataset['train']) // effective_batch_size)  # ceil division
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, int(total_steps * WARMUP_RATIO))

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    logging_steps=25,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=False,  # T4 (Turing) has no native bf16 Tensor Core support
    fp16=True,   # use fp16 mixed precision instead, much faster on T4
    max_length=1024,
    dataset_text_field='text',
    packing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    peft_config=lora_config,
)

# peft/trl bug workaround: get_peft_model() reads model.config.torch_dtype (bfloat16 in
# Qwen2.5's config.json) to pick the LoRA adapter dtype, ignoring the fp16 weights we
# actually loaded — so adapters end up bf16. The fp16 GradScaler additionally requires
# trainable (master) params to be fp32 to safely unscale gradients, so cast to fp32
# (not fp16): forward/backward still runs in fp16 under autocast either way.
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## Merge LoRA adapter into the base model (needed before GGUF conversion)

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto')
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(f'{OUTPUT_DIR}-merged')
tokenizer.save_pretrained(f'{OUTPUT_DIR}-merged')

## Convert to GGUF (run in Colab)


In [ ]:
!test -d llama.cpp || git clone -q https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!mkdir -p models

# Workaround: some transformers versions choke on tokenizer_config.json's
# "extra_special_tokens" being a list instead of a dict (AttributeError: 'list'
# object has no attribute 'keys'). Normalize it before AutoTokenizer loads it.
import json, os
tok_cfg_path = f"{OUTPUT_DIR}-merged/tokenizer_config.json"
with open(tok_cfg_path) as f:
    tok_cfg = json.load(f)
if isinstance(tok_cfg.get("extra_special_tokens"), list):
    tok_cfg["extra_special_tokens"] = {}
    with open(tok_cfg_path, "w") as f:
        json.dump(tok_cfg, f, indent=2)
    print("Patched extra_special_tokens in", tok_cfg_path)

# Step 1: convert the merged HF model to an intermediate f16 GGUF
# (recent llama.cpp versions dropped direct q4_k_m output from this script)
!python llama.cpp/convert_hf_to_gguf.py {OUTPUT_DIR}-merged --outfile models/text2sql-qwen2.5-1.5b-f16.gguf --outtype f16

# Step 2: build llama-quantize and use it to produce the q4_k_m version
!cmake -S llama.cpp -B llama.cpp/build -DBUILD_SHARED_LIBS=OFF
!cmake --build llama.cpp/build --target llama-quantize -j $(nproc)
!./llama.cpp/build/bin/llama-quantize models/text2sql-qwen2.5-1.5b-f16.gguf models/text2sql-qwen2.5-1.5b.gguf q4_k_m

# Download the quantized .gguf file to your local machine (e.g. via the Colab Files panel,
# or files.download() below), then follow the 'Register with Ollama' steps locally.
from google.colab import files
files.download('models/text2sql-qwen2.5-1.5b.gguf')

## Register with Ollama (run locally, NOT in Colab)
Ollama runs on your own machine, not in the Colab cloud VM. After downloading the `.gguf` file above:
1. Move it into `models/` in this project (next to `models/Modelfile`).
2. In a local terminal:
```bash
cd models
ollama create text2sql-qwen2.5-1.5b -f Modelfile
ollama run text2sql-qwen2.5-1.5b
```